# Laboratorio 3 - Milton Beltrán

**Clasificación de texto con Naive Bayes** · Corpus: Spanish News Classification (`df_total.csv`)

## 0. Preparación del corpus

In [ ]:
# --- Manejo de datos y gráficos ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns                   


import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer


from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer


from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [ ]:
# --- Carga del corpus y limpieza de filas inservibles ---

df = pd.read_csv("df_total.csv")

#Filas duplicadas Se eliminan antes de normalizar.
df_prep = df.drop_duplicates().reset_index(drop=True)
print("Documentos originales:", len(df))
print("Documentos tras quitar duplicados:", len(df_prep))


vacios = df_prep["news"].str.strip() == ""
print("Documentos con texto vacío:", int(vacios.sum()), "->", df_prep.loc[vacios, "Type"].tolist())

df_prep = df_prep[~vacios].reset_index(drop=True)
print("Documentos de trabajo:", len(df_prep))
print("Categorías:", df_prep["Type"].nunique())

print()
print(df_prep["Type"].value_counts())

In [ ]:

contradictorios = df_prep.duplicated("news", keep=False)
print("Filas con texto repetido y etiqueta contradictoria:", int(contradictorios.sum()))
print(df_prep.loc[contradictorios, "Type"].value_counts().to_string())

df_prep = df_prep[~contradictorios].reset_index(drop=True)
print()
print("Documentos de trabajo (definitivo):", len(df_prep))
print(df_prep["Type"].value_counts().to_string())

In [ ]:
# Pipeline de normalización (heredado de los Laboratorios #1 y #2)
# tokenización -> minúsculas -> sin puntuación -> sin stopwords -> stemming


stop_es = set(stopwords.words("spanish"))
stemmer = SnowballStemmer("spanish")

# i. Tokenización:
df_prep["tokens"] = df_prep["news"].apply(
    lambda t: word_tokenize(t, language="spanish")
)

# ii. Minúsculas
df_prep["tokens"] = df_prep["tokens"].apply(
    lambda lista: [tok.lower() for tok in lista]
)

# iii. Quitar puntuación
df_prep["tokens"] = df_prep["tokens"].apply(
    lambda lista: [tok for tok in lista if tok.isalpha()]
)

# iv. Quitar stopwords
df_prep["tokens"] = df_prep["tokens"].apply(
    lambda lista: [tok for tok in lista if tok not in stop_es]
)

# v. Stemming ("lematización"): 
df_prep["tokens_norm"] = df_prep["tokens"]

df_prep["tokens_stem"] = df_prep["tokens_norm"].apply(
    lambda lista: [stemmer.stem(tok) for tok in lista]
)


def contar(col_de_listas):
    """Aplana una columna de listas de tokens y devuelve (tokens, tipos).

    - tokens: total de ocurrencias (con repeticiones).
    - tipos:  palabras distintas (el vocabulario).
    """
    todos = [tok for lista in col_de_listas for tok in lista]
    return len(todos), len(set(todos))


tok, tip = contar(df_prep["tokens_stem"])
print(f"Corpus normalizado: documentos={len(df_prep):,}  tokens={tok:,}  tipos={tip:,}")
print("Ejemplo doc 0, legible:", df_prep["tokens_norm"].iloc[0][:12])
print("Ejemplo doc 0, raíces:", df_prep["tokens_stem"].iloc[0][:12])

In [ ]:
# Del corpus tokenizado al formato que espera scikit-learn
df_prep["texto_norm"] = df_prep["tokens_stem"].apply(lambda lista: " ".join(lista))

print("Documentos listos para vectorizar:", len(df_prep))
print("Columnas del DataFrame:", list(df_prep.columns))
print()
print("Documento 0, primeros 300 caracteres:")
print(df_prep["texto_norm"].iloc[0][:300])
print()
print("Categoría del documento 0:", df_prep["Type"].iloc[0])

## 1. Conjuntos de entrenamiento, validación y prueba

División estratificada del corpus normalizado en 70 % entrenamiento, 15 % validación y 15 % prueba.

In [ ]:
# --- 1.1 División estratificada 70 / 15 / 15 ---
# train_test_split solo parte en DOS, así que se llama dos veces:
#   1) corpus -> train (70 %) + temp (30 %)
#   2) temp   -> validación (15 %) + prueba (15 %)   <- test_size=0.5 sobre el 30 %, no 0.15

RANDOM_STATE = 42  

y_total = df_prep["Type"]

idx_train, idx_temp = train_test_split(
    df_prep.index,
    test_size=0.30,
    stratify=y_total,               
    random_state=RANDOM_STATE,
)

idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50,                 # la mitad del 30 % -> 15 % y 15 %
    stratify=y_total.loc[idx_temp], 
    random_state=RANDOM_STATE,
)

# Textos normalizados y etiquetas de cada conjunto.
X_train_txt, y_train = df_prep.loc[idx_train, "texto_norm"], y_total.loc[idx_train]
X_val_txt,   y_val   = df_prep.loc[idx_val,   "texto_norm"], y_total.loc[idx_val]
X_test_txt,  y_test  = df_prep.loc[idx_test,  "texto_norm"], y_total.loc[idx_test]

total = len(df_prep)
for nombre, conj in [("Entrenamiento", y_train), ("Validación", y_val), ("Prueba", y_test)]:
    print(f"{nombre:<15}{len(conj):>5} documentos  ({len(conj)/total:.1%})")
print(f"{'Total':<15}{len(y_train) + len(y_val) + len(y_test):>5} documentos")

# Los tres conjuntos deben ser disjuntos y cubrir el corpus completo.
assert len(set(idx_train) & set(idx_val)) == 0
assert len(set(idx_train) & set(idx_test)) == 0
assert len(set(idx_val) & set(idx_test)) == 0
assert len(idx_train) + len(idx_val) + len(idx_test) == total

In [ ]:
# --- 1.2 Distribución de categorías en cada conjunto ---
dist = pd.DataFrame({
    "Entrenamiento": y_train.value_counts(),
    "Validación": y_val.value_counts(),
    "Prueba": y_test.value_counts(),
}).fillna(0).astype(int)

dist = dist.loc[y_total.value_counts().index]   # ordenar de la categoría más frecuente a la menos
dist["Total"] = dist.sum(axis=1)

pct = (dist[["Entrenamiento", "Validación", "Prueba"]]
       / dist[["Entrenamiento", "Validación", "Prueba"]].sum()) * 100

tabla = dist.copy()
for c in ["Entrenamiento", "Validación", "Prueba"]:
    tabla[c] = [f"{n} ({p:.1f}%)" for n, p in zip(dist[c], pct[c])]

print("Documentos por categoría y conjunto (porcentaje dentro de cada conjunto)")
print(tabla.to_string())
print()
print("Categorías ausentes en algún conjunto:", int((dist[["Entrenamiento", "Validación", "Prueba"]] == 0).sum().sum()))
print(f"Desviación máxima entre proporciones: {(pct.max(axis=1) - pct.min(axis=1)).max():.2f} puntos porcentuales")

In [ ]:
# --- 1.3 Gráfico de la distribución ---
# Se grafican PORCENTAJES
ax = pct.plot(kind="bar", figsize=(9, 4), width=0.78,
              color=["#0b3d5c", "#16679a", "#8ab6d6"], edgecolor="white")

ax.set_title("Distribución de categorías por conjunto (estratificada)")
ax.set_xlabel("")
ax.set_ylabel("% dentro del conjunto")
ax.legend(title=None, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("reporte/img/particiones.png", dpi=120)
plt.show()

### 1.4 Resultados y discusión

El corpus de 1,134 documentos quedó dividido en **794 / 170 / 170** (70 % / 15 % / 15 %).
La estratificación no tuvo un efecto relevante en la proporción de cada categoría en los tres
conjuntos, con una desviación máxima inferior a un punto porcentual, y ninguna categoría
desapareció de ningún conjunto.

#### ¿Para qué sirve cada conjunto?

- **Entrenamiento (794 docs).** Es lo único que el modelo puede mirar para aprender. De aquí
  salen las probabilidades del clasificador, cuántas noticias hay de cada categoría y con qué
  frecuencia aparece cada palabra dentro de ellas.

- **Validación (170 docs).** Es el entorno  de pruebas durante el desarrollo. Sirve para comparar
  alternativas (BoW contra TF-IDF, vocabulario completo contra `max_features`, etc.)
  y quedarse con la mejor. Son datos que el modelo no vio al entrenar, así
  que la medición es honesta; pero al usarlos repetidamente para elegir, las decisiones se
  van distorcionando por un fenómeno de "memorizar los datos".

- **Prueba (170 docs).** Es la prueba final y se hace **una sola vez**, al terminar. Su
  número no sirve para decidir nada: sirve para estimar cómo se comportaría el clasificador
  con noticias nuevas.

#### ¿Por qué no usar el conjunto de prueba para tomar decisiones?

Porque en el momento en que un resultado de prueba cambia lo que uno hace, ese conjunto deja
de ser datos no vistos. Si midiéramos en prueba, viéramos que TF-IDF gana y por eso
eligiéramos TF-IDF, la información de prueba ya influenció al modelo final o a través de nuestras decisiones.
 La cifra resultante seguiría siendo alta, pero ya no mediría generalización;
 sino qué tan bien ajustamos el modelo a esos 170
documentos concretos.

## 2. Construcción del clasificador Naive Bayes

Dos modelos `MultinomialNB` sobre las mismas particiones: uno alimentado con Bolsa de Palabras
y otro con TF-IDF. Los vectorizadores se ajustan **solo con el conjunto de entrenamiento**.

In [ ]:
# --- 2.1 Bolsa de palabras ajustada SOLO con entrenamiento ---
bow_vectorizer = CountVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train_txt)   # fit + transform: SOLO entrenamiento
X_val_bow   = bow_vectorizer.transform(X_val_txt)         # transform: sin aprender nada nuevo
X_test_bow  = bow_vectorizer.transform(X_test_txt)

vocab_bow = bow_vectorizer.get_feature_names_out()
print("Matriz de entrenamiento:", X_train_bow.shape)
print("Matriz de validación:   ", X_val_bow.shape)
print("Matriz de prueba:       ", X_test_bow.shape)
print("Vocabulario aprendido en entrenamiento:", f"{len(vocab_bow):,} términos")

# Cuánto vocabulario se habría "colado" al ajustar con todo el corpus (solo para dimensionar la fuga).
vocab_corpus = CountVectorizer().fit(df_prep["texto_norm"]).get_feature_names_out()
print("Vocabulario si se ajustara con el corpus completo:", f"{len(vocab_corpus):,} términos",
      f"(+{len(vocab_corpus) - len(vocab_bow):,})")

# Palabras fuera de vocabulario (OOV) en validación: las que el modelo simplemente no puede ver.
analizador = bow_vectorizer.build_analyzer()
tokens_val = [t for texto in X_val_txt for t in analizador(texto)]
oov = [t for t in tokens_val if t not in bow_vectorizer.vocabulary_]
print(f"\nTokens en validación: {len(tokens_val):,}")
print(f"Ocurrencias fuera de vocabulario: {len(oov):,} ({len(oov)/len(tokens_val):.2%}) "
      f"| términos distintos: {len(set(oov)):,}")

In [ ]:
# --- 2.2 TF-IDF ajustado de la misma forma ---
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_txt)
X_val_tfidf   = tfidf_vectorizer.transform(X_val_txt)
X_test_tfidf  = tfidf_vectorizer.transform(X_test_txt)

vocab_tfidf = tfidf_vectorizer.get_feature_names_out()
print("Matriz TF-IDF de entrenamiento:", X_train_tfidf.shape)
print("¿Mismo vocabulario que BoW?", list(vocab_bow) == list(vocab_tfidf))
print()

# Misma forma, contenido distinto: BoW guarda conteos enteros, TF-IDF pesos reales normalizados.
i = 0
fila_bow, fila_tfidf = X_train_bow[i], X_train_tfidf[i]
print(f"Documento de entrenamiento {i} — términos distintos: {fila_bow.nnz}")
print("  BoW  : valores enteros, máximo =", int(fila_bow.data.max()))
print(f"  TFIDF: valores reales,  máximo = {fila_tfidf.data.max():.4f}, "
      f"norma L2 = {np.sqrt((fila_tfidf.data ** 2).sum()):.4f}")

In [ ]:
# --- 2.3 Entrenamiento de los dos clasificadores ---
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)

nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

# --- Lo que aprendió el modelo, pieza por pieza ---
# class_log_prior_  -> log P(c)
# feature_log_prob_ -> log P(w|c), una fila por clase y una columna por término del vocabulario
priors = pd.DataFrame({
    "Documentos en train": nb_bow.class_count_.astype(int),
    "P(c)": np.exp(nb_bow.class_log_prior_),
}, index=nb_bow.classes_).sort_values("P(c)", ascending=False)

print("Probabilidades a priori estimadas desde el conjunto de entrenamiento:")
print(priors.to_string(formatters={"P(c)": "{:.4f}".format}))
print()
print("Matriz de verosimilitudes log P(w|c):", nb_bow.feature_log_prob_.shape,
      "-> (clases, términos del vocabulario)")
# Cada fila es una distribución de probabilidad sobre el vocabulario completo: debe sumar 1.
print("Suma de P(w|c) sobre el vocabulario, por clase:",
      np.round(np.exp(nb_bow.feature_log_prob_).sum(axis=1), 6))

In [ ]:
# --- 2.4 Cómo se ve P(w|c) para palabras concretas ---
raices = ["inflacion", "alianz", "reput", "sostenibil", "emision", "banc"]

filas = []
for r in raices:
    if r in bow_vectorizer.vocabulary_:
        j = bow_vectorizer.vocabulary_[r]
        filas.append(pd.Series(np.exp(nb_bow.feature_log_prob_[:, j]),
                               index=nb_bow.classes_, name=r))

verosim = pd.DataFrame(filas)
print("P(w|c) — probabilidad de cada raíz dentro de cada categoría")
print((verosim * 1000).round(2).to_string())
print("(valores × 1000 para que se lean; cada fila es una raíz, cada columna una categoría)")
print()
for r in verosim.index:
    ganadora = verosim.loc[r].idxmax()
    razon = verosim.loc[r].max() / verosim.loc[r].drop(ganadora).max()
    print(f"  '{r}': más probable en {ganadora} ({razon:.1f}× la siguiente categoría)")

### 2.5 El teorema de Bayes sobre este corpus

Naive Bayes no compara documentos entre sí,
sino que se pregunta qué tan probable es cada categoría dado el texto que está leyendo. El
problema es que P(c|d) no se puede contar directamente: no tenemos forma de saber cuántas veces
"este documento exacto" resultó ser de Macroeconomia. Bayes nos deja darle la vuelta y
expresarlo con cosas que sí se pueden contar, P(c|d) = P(d|c) · P(c) / P(d). El denominador
P(d) es el mismo para las 7 categorías. Quedan dos
piezas, y las dos se estiman contando sobre el conjunto de entrenamiento.

**P(c)** es qué tan común es cada categoría antes de leer nada. Aquí sale de la frecuencia de
clases en los 793 documentos de entrenamiento: Macroeconomia son 223 (0.281) y Reputacion solo
18 (0.023).  **P(wᵢ|c)** es qué tan probable es que una palabra aparezca dentro de las noticias de esa
categoría, es decir qué proporción ocupa esa palabra entre todas las palabras de la categoría
en entrenamiento. En el modelo entrenado esto es la matriz `feature_log_prob_`, de 7 × 10,968:
una distribución completa sobre el vocabulario por cada categoría. 

En la celda 2.4 se ve la diferencia entre una palabra útil y una que no. `inflacion` vale
0.0126 en Macroeconomia y 29 veces menos en la siguiente categoría, así que cada vez que
aparece empuja fuerte hacia esa clase. `banc`, en cambio, tiene un valor parecido en las siete
categorías y su mejor opción apenas supera a la segunda por 1.5×: aparece mucho, pero no
discrimina, porque el corpus entero habla de bancos.

Para clasificar una noticia nueva, el modelo calcula para cada categoría

score(c) = log P(c) + Σᵢ nᵢ · log P(wᵢ|c)

donde nᵢ es cuántas veces aparece la palabra wᵢ en el documento, y se queda con el score más
alto. Se usan logaritmos porque multiplicar cientos de probabilidades pequeñas termina dando
cero por precisión de la máquina; al sumar logaritmos el resultado es equivalente y el orden de
las categorías no cambia. 

## 3. Entrenamiento y evaluación

Los dos modelos se miden sobre **validación**. El conjunto de prueba no se toca en esta
sección: se reserva para la evaluación final, una vez tomadas todas las decisiones
(Secciones 5 y 6).

In [ ]:
# --- 3.1 Métricas de los dos modelos en entrenamiento y validación ---
# Se mide también sobre entrenamiento, no porque ese número valga como desempeño (el modelo
# ya vio esos documentos), sino porque la BRECHA entre train y validación es lo que revela
# sobreajuste.
def resumen(modelo, X, y):
    """Devuelve accuracy y F1 macro/ponderado de un modelo sobre un conjunto."""
    pred = modelo.predict(X)
    return {
        "accuracy": accuracy_score(y, pred),
        "F1 macro": f1_score(y, pred, average="macro"),
        "F1 ponderado": f1_score(y, pred, average="weighted"),
    }


filas = {
    ("BoW", "entrenamiento"): resumen(nb_bow, X_train_bow, y_train),
    ("BoW", "validación"): resumen(nb_bow, X_val_bow, y_val),
    ("TF-IDF", "entrenamiento"): resumen(nb_tfidf, X_train_tfidf, y_train),
    ("TF-IDF", "validación"): resumen(nb_tfidf, X_val_tfidf, y_val),
}

metricas = pd.DataFrame(filas).T.round(4)
metricas.index.names = ["Representación", "Conjunto"]
print(metricas.to_string())

print()
for nombre in ["BoW", "TF-IDF"]:
    brecha = metricas.loc[(nombre, "entrenamiento"), "accuracy"] - metricas.loc[(nombre, "validación"), "accuracy"]
    print(f"{nombre:<7} brecha entrenamiento - validación: {brecha:.3f} ({brecha:.1%})")

In [ ]:
# --- 3.2 Reporte por categoría en validación ---
# classification_report da precision, recall y F1 de cada clase, más los dos promedios:
#   macro     -> promedia las 7 clases con el mismo peso (Reputacion pesa igual que Macroeconomia)
#   ponderado -> pondera por número de documentos (las clases grandes dominan)
pred_val_bow = nb_bow.predict(X_val_bow)
pred_val_tfidf = nb_tfidf.predict(X_val_tfidf)

print("=" * 62)
print("BoW — validación")
print("=" * 62)
print(classification_report(y_val, pred_val_bow, zero_division=0))

print("=" * 62)
print("TF-IDF — validación")
print("=" * 62)
print(classification_report(y_val, pred_val_tfidf, zero_division=0))

# ¿Cómo reparte cada modelo sus predicciones? Si un modelo colapsa hacia la clase mayoritaria,
# se ve aquí antes que en cualquier métrica.
reparto = pd.DataFrame({
    "Documentos reales": y_val.value_counts(),
    "Predichos por BoW": pd.Series(pred_val_bow).value_counts(),
    "Predichos por TF-IDF": pd.Series(pred_val_tfidf).value_counts(),
}).fillna(0).astype(int).loc[y_val.value_counts().index]
print("Reparto de las 170 predicciones de validación")
print(reparto.to_string())

In [ ]:
# --- 3.3 Matrices de confusión ---
# Filas = categoría real, columnas = categoría predicha. La diagonal son los aciertos;
# todo lo que se sale de ella es un error, y su posición dice con qué se confundió.
clases = nb_bow.classes_

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for ax, (titulo, pred) in zip(axes, [("BoW", pred_val_bow), ("TF-IDF", pred_val_tfidf)]):
    cm = confusion_matrix(y_val, pred, labels=clases)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=clases, yticklabels=clases, ax=ax,
                linewidths=0.5, linecolor="white")
    acc = accuracy_score(y_val, pred)
    ax.set_title(f"{titulo} — validación (accuracy {acc:.2f})")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig("reporte/img/confusion_validacion.png", dpi=120)
plt.show()

In [ ]:
# --- 3.4 Por qué TF-IDF se derrumba: prior contra evidencia ---
# El score de cada clase es log P(c) + suma de los log P(w|c) del documento. Si la evidencia
# léxica es grande, el prior es irrelevante; si es pequeña, el prior manda y el modelo se va
# hacia la clase mayoritaria. Se mide cuánto pesa cada parte en los dos modelos:
#   - "evidencia": diferencia entre la clase mejor y la peor puntuada, promediada sobre los
#      documentos de validación (cuánto separan las palabras a las clases).
#   - "prior": diferencia entre log P(c) de la clase más y la menos frecuente (fija: 2.52).
print(f"Suma de valores por documento (media en validación)")
print(f"  BoW   : {np.asarray(X_val_bow.sum(axis=1)).ravel().mean():>8.1f}  (conteos de palabras)")
print(f"  TF-IDF: {np.asarray(X_val_tfidf.sum(axis=1)).ravel().mean():>8.2f}  (pesos normalizados a norma 1)")
print()

for nombre, modelo, X in [("BoW", nb_bow, X_val_bow), ("TF-IDF", nb_tfidf, X_val_tfidf)]:
    evidencia = np.asarray(X @ modelo.feature_log_prob_.T)      # log-verosimilitud por clase
    rango_evidencia = (evidencia.max(axis=1) - evidencia.min(axis=1)).mean()
    rango_prior = modelo.class_log_prior_.max() - modelo.class_log_prior_.min()
    print(f"{nombre:<7} evidencia léxica: {rango_evidencia:>7.2f} | prior: {rango_prior:.2f} "
          f"| la evidencia pesa {rango_evidencia / rango_prior:.1f}× más que el prior")

### 3.5 Resultados y discusión

| Representación | Conjunto | Accuracy | F1 macro | F1 ponderado |
|---|---|---|---|---|
| BoW | entrenamiento | 0.941 | 0.936 | 0.941 |
| **BoW** | **validación** | **0.800** | **0.732** | **0.795** |
| TF-IDF | entrenamiento | 0.710 | 0.548 | 0.663 |
| TF-IDF | validación | 0.547 | 0.360 | 0.474 |

#### ¿Hay sobreajuste?

En BoW la brecha entre entrenamiento (0.941) y validación (0.800) es de **14 puntos**. Algo de
sobreajuste hay, y era esperable: el modelo tiene 10,968 features y solo 793 documentos para
aprenderlas, así que muchas palabras aparecen en muy pocos documentos y el modelo termina
memorizando parte del vocabulario de entrenamiento. Aun así no es un caso grave: el desempeño
en validación se sostiene en 0.80 y no se desploma, que es lo que pasaría si el modelo hubiera
memorizado sin aprender nada general. El suavizado de Laplace ayuda justamente ahí, porque
impide que una palabra rara vista una sola vez decida sola una categoría.

En TF-IDF la brecha es parecida (0.710 contra 0.547, 16 puntos), pero el problema no es
sobreajuste: es que el modelo funciona mal **también** en entrenamiento. Un modelo que ni
siquiera aprende bien los datos que vio no está sobreajustado, está mal planteado.

#### ¿Cuál modelo ganó y por qué?

BoW gana por 25 puntos de accuracy y 37 de F1 macro. El reparto de predicciones de la celda 3.2
muestra qué le pasa a TF-IDF: de 170 documentos de validación, predice **109 veces
Macroeconomia**, que es la clase más grande. Deja Reputacion en cero, Otra en 1 y Regulaciones
en 2. No está clasificando, está apostando a la clase mayoritaria.

La celda 3.4 explica por qué. El score de cada clase es log P(c) más la suma de las
log-verosimilitudes de las palabras del documento, así que lo que decide es cuál de los dos
términos pesa más. En BoW cada documento aporta en promedio **261 conteos**, y la diferencia de
evidencia entre la mejor y la peor clase es de 242 puntos de log-probabilidad, contra un prior
cuyo rango completo es de 2.52: la evidencia pesa **96 veces más**, el prior es irrelevante y
gana quien tenga las palabras. En TF-IDF los pesos están normalizados a norma L2 = 1, así que
cada documento suma apenas **9.9** en lugar de 261, y la evidencia se reduce a 4.76 puntos:
apenas **1.9 veces** el prior. Con esa proporción, el prior de Macroeconomia (0.281 contra
0.023 de Reputacion) alcanza para inclinar la decisión en cuanto el texto no es clarísimo.

Dicho de otra forma: TF-IDF sí pondera mejor las palabras —le baja el peso a las que salen en
todos los documentos, como `banc`— pero al normalizar destruye la magnitud de los conteos, que
es justo la información que `MultinomialNB` necesita para que la evidencia domine sobre el
prior. La representación que era la mejor para medir similitud en el Laboratorio #2 es la peor
para este clasificador, porque cada modelo necesita una cosa distinta de la representación.

#### Lo que se ve por categoría

En BoW, Innovacion tiene recall 1.00 pero precision 0.66: recibe 35 predicciones cuando solo
hay 23 documentos suyos, así que funciona como imán de documentos ajenos. Otra es el caso
contrario, precision 1.00 y recall 0.58: cuando el modelo dice "Otra" acierta siempre, pero se
le escapan 8 de 19. Reputacion, con 4 documentos en validación, acierta 1: su F1 de 0.40 no es
una estimación confiable de nada. La confusión mutua más frecuente es **Alianzas ↔
Regulaciones**, con 7 errores cruzados, y es la que se analiza en la Sección 4.